**Clean Persona data**


In [1]:
import re

# File paths in Google Colab environment
input_file_path = '/content/Keng-Training data. uncleaned.txt'
output_file_path = '/content/keng_dataset_optionA.txt'

def process_discord_option_a(input_path, output_path):
    with open(input_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    messages = []
    current_block = []

    # Regex patterns for Discord headers, line numbers, and timestamps
    # 1. Matches timestamps like 27/12/2565 16:13 or 3/6/2565 18:59
    timestamp_pattern = re.compile(r'\b\d{1,2}/\d{1,2}/\d{4}\s+\d{1,2}:\d{2}\b')

    # 2. Matches headers like "Keng —", "Deleted User —", or numbered prefixes like "1. Deleted User —"
    header_pattern = re.compile(r'^(\d+\.\s*)?(Keng|Deleted User)(\s*—)?', re.IGNORECASE)

    for line in lines:
        cleaned_line = line.strip()

        # Skip completely empty lines
        if not cleaned_line:
            continue

        # Check if line is a timestamp, username header, or numbered list header
        if (timestamp_pattern.search(cleaned_line) or
            header_pattern.match(cleaned_line) or
            re.match(r'^\d+\.$', cleaned_line)):

            # If we hit a metadata line and have accumulated text, flush it as a complete message
            if current_block:
                full_text = " ".join(current_block).strip()
                if full_text:
                    messages.append(full_text)
                current_block = []
        else:
            # It's actual message text from Keng/Deleted User
            current_block.append(cleaned_line)

    # Flush any remaining message at the end of the file
    if current_block:
        full_text = " ".join(current_block).strip()
        if full_text:
            messages.append(full_text)

    # Write out using Option A (<|endoftext|> delimiter per entry)
    with open(output_path, 'w', encoding='utf-8') as f:
        for msg in messages:
            f.write(f"{msg} <|endoftext|>\n")

    print(f"Successfully processed {len(messages)} entries!")
    print(f"Cleaned dataset saved to: {output_path}")

# Run the cleaner script
process_discord_option_a(input_file_path, output_file_path)

Successfully processed 212 entries!
Cleaned dataset saved to: /content/keng_dataset_optionA.txt


In [2]:
import os
import pandas as pd
from datasets import load_dataset
import kagglehub

# ---------------------------------------------------------
# 1. TinyStories (Skip if already downloaded)
# ---------------------------------------------------------
tinystories_csv = "/content/tinystories_raw.csv"
if os.path.exists(tinystories_csv):
    print("=== 1. TinyStories already exists. Loading from CSV... ===")
    df_tinystories = pd.read_csv(tinystories_csv)
else:
    print("=== 1. Downloading TinyStories ===")
    ts_ds = load_dataset("roneneldan/TinyStories", split="train")
    df_tinystories = pd.DataFrame(ts_ds)
    df_tinystories.to_csv(tinystories_csv, index=False)

print("TinyStories shape:", df_tinystories.shape)

# ---------------------------------------------------------
# 2. Thai Wikipedia (Skip if already downloaded)
# ---------------------------------------------------------
thai_wiki_csv = "/content/thai_wiki_raw.csv"
if os.path.exists(thai_wiki_csv):
    print("\n=== 2. Thai Wiki already exists. Loading from CSV... ===")
    df_thai_wiki = pd.read_csv(thai_wiki_csv)
else:
    print("\n=== 2. Downloading Thai Wiki ===")
    thai_wiki_ds = load_dataset("pythainlp/thai_wikipedia_clean_20230101", split="train")
    df_thai_wiki = pd.DataFrame(thai_wiki_ds)
    df_thai_wiki.to_csv(thai_wiki_csv, index=False)

print("Thai Wiki shape:", df_thai_wiki.shape)

# ---------------------------------------------------------
# 3. SCB Thai-English (Kaggle Download Fix)
# ---------------------------------------------------------
scb_csv = "/content/scb_raw.csv"
if os.path.exists(scb_csv):
    print("\n=== 3. SCB MT EN-TH already exists. Loading from CSV... ===")
    df_scb = pd.read_csv(scb_csv)
else:
    print("\n=== 3. Downloading SCB Thai-English from Kaggle ===")
    # Download dataset directory from Kaggle
    scb_path = kagglehub.dataset_download("ratthachat/scb-mt-enth-2020")

    # Locate downloaded CSV/JSON file inside directory
    downloaded_files = os.listdir(scb_path)
    print("Files found in SCB Kaggle download:", downloaded_files)

    target_file = None
    for f in downloaded_files:
        if f.endswith('.csv') or f.endswith('.json') or f.endswith('.parquet'):
            target_file = os.path.join(scb_path, f)
            break

    if target_file and target_file.endswith('.csv'):
        df_scb = pd.read_csv(target_file)
    elif target_file and target_file.endswith('.json'):
        df_scb = pd.read_json(target_file)
    elif target_file and target_file.endswith('.parquet'):
        df_scb = pd.read_parquet(target_file)
    else:
        # Fallback reading
        first_file = os.path.join(scb_path, downloaded_files[0])
        df_scb = pd.read_csv(first_file)

    df_scb.to_csv(scb_csv, index=False)

print("SCB shape:", df_scb.shape)

# ---------------------------------------------------------
# 4. DailyDialog (Kaggle Download Fix)
# ---------------------------------------------------------
dailydialog_csv = "/content/dailydialog_raw.csv"
if os.path.exists(dailydialog_csv):
    print("\n=== 4. DailyDialog already exists. Loading from CSV... ===")
    df_dailydialog = pd.read_csv(dailydialog_csv)
else:
    print("\n=== 4. Downloading DailyDialog from Kaggle ===")
    dd_path = kagglehub.dataset_download("thedevastator/dailydialog-unlock-the-conversation-potential-in")

    downloaded_files = os.listdir(dd_path)
    print("Files found in DailyDialog Kaggle download:", downloaded_files)

    target_file = None
    for f in downloaded_files:
        if f.endswith('.csv'):
            target_file = os.path.join(dd_path, f)
            break

    if target_file:
        df_dailydialog = pd.read_csv(target_file)
    else:
        first_file = os.path.join(dd_path, downloaded_files[0])
        df_dailydialog = pd.read_csv(first_file)

    df_dailydialog.to_csv(dailydialog_csv, index=False)

print("DailyDialog shape:", df_dailydialog.shape)

# ---------------------------------------------------------
# PRINT HEADS FOR INSPECTION
# ---------------------------------------------------------
print("\n================ DATASET HEADS ================")
print("\n--- 1. TinyStories Head ---")
print(df_tinystories.head(2))

print("\n--- 2. Thai Wiki Head ---")
print(df_thai_wiki.head(2))

print("\n--- 3. SCB MT EN-TH Head ---")
print(df_scb.head(2))

print("\n--- 4. DailyDialog Head ---")
print(df_dailydialog.head(2))

=== 1. Downloading TinyStories ===


README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…): reconstructing file:   0%|          |  0.00B /  249MB            

data/train-00000-of-00004-2d5a1467fff108(…): downloading bytes:           |  0.00B            

data/train-00001-of-00004-5852b56a2bd28f(…): reconstructing file:   0%|          |  0.00B /  248MB            

data/train-00001-of-00004-5852b56a2bd28f(…): downloading bytes:           |  0.00B            

... (17 outputs left out)

Extracting files...


Files found in SCB Kaggle download: ['scb_mt_enth_2020_validation.csv', 'scb_mt_enth_2020_test.csv', 'scb_mt_enth_2020_train.csv']
SCB shape: (100173, 4)

=== 4. Downloading DailyDialog from Kaggle ===


100%|██████████| 2.46M/2.46M [00:00<00:00, 170MB/s]

Extracting files...
Files found in DailyDialog Kaggle download: ['test.csv', 'validation.csv', 'train.csv']
DailyDialog shape: (1000, 3)

================ DATASET HEADS ================

--- 1. TinyStories Head ---
                                                text
0  One day, a little girl named Lily found a need...
1  Once upon a time, there was a little car named...

--- 2. Thai Wiki Head ---
                                                text
0                                          พาร์สลีย์
1  พาร์สลีย์  เป็นพืชดอกในวงศ์ผักชี (Apiaceae) มี...

--- 3. SCB MT EN-TH Head ---
   Unnamed: 0                    subdataset  \
0           0  generated_reviews_translator   
1           1          generated_reviews_yn   

                                                  en  \
0     I have many memories that cannot be recovered.   
1  I loved the look of this item but when we actu...   

                                                  th  
0  ฉันมีความทรงจําที่เก็บไว้หลายอย่างมากที่ก

In [3]:
import pandas as pd
import ast
import random
import os

print("=== Step 1: Loading & Boosting Keng's Data ===")
keng_lines = []
if os.path.exists('/content/keng_dataset_optionA.txt'):
    with open('/content/keng_dataset_optionA.txt', 'r', encoding='utf-8') as f:
        keng_lines = [line.strip() for line in f if line.strip()]
    # Repeat Keng's dataset 10x so the small model encounters his slang/personality frequently
    keng_boosted = keng_lines * 10
    print(f"Loaded {len(keng_lines)} Keng lines -> Boosted 10x to {len(keng_boosted)} total entries.")
else:
    print("Warning: /content/keng_dataset_optionA.txt not found!")
    keng_boosted = []

print("\n=== Step 2: Processing TinyStories ===")
df_ts = pd.read_csv('/content/tinystories_raw.csv')
# Sample 100,000 entries to prevent over-dominating the dataset
ts_sample = df_ts['text'].dropna().sample(n=min(100000, len(df_ts)), random_state=42)
ts_lines = [f"{str(text).strip()} <|endoftext|>" for text in ts_sample]
print(f"Prepared {len(ts_lines)} TinyStories entries.")

print("\n=== Step 3: Processing Thai Wiki ===")
df_wiki = pd.read_csv('/content/thai_wiki_raw.csv')
# Filter out short text/titles (< 40 characters) like "พาร์สลีย์"
wiki_clean = df_wiki['text'].dropna()
wiki_clean = wiki_clean[wiki_clean.str.len() > 40]
wiki_sample = wiki_clean.sample(n=min(50000, len(wiki_clean)), random_state=42)
wiki_lines = [f"{str(text).strip()} <|endoftext|>" for text in wiki_sample]
print(f"Prepared {len(wiki_lines)} Thai Wiki entries.")

print("\n=== Step 4: Processing SCB Thai-English ===")
df_scb = pd.read_csv('/content/scb_raw.csv')
scb_sample = df_scb.dropna(subset=['en', 'th']).sample(n=min(50000, len(df_scb)), random_state=42)
scb_lines = []
for _, row in scb_sample.iterrows():
    scb_lines.append(f"{str(row['en']).strip()} <|endoftext|>")
    scb_lines.append(f"{str(row['th']).strip()} <|endoftext|>")
print(f"Prepared {len(scb_lines)} SCB entries (English & Thai).")

print("\n=== Step 5: Processing DailyDialog ===")
df_dd = pd.read_csv('/content/dailydialog_raw.csv')
dd_lines = []
for dialog_str in df_dd['dialog'].dropna():
    try:
        # Safely parse stringified python list: "['line 1', 'line 2']"
        turns = ast.literal_eval(dialog_str)
        if isinstance(turns, list) and len(turns) > 0:
            conversation = "\n".join([turn.strip() for turn in turns])
            dd_lines.append(f"{conversation} <|endoftext|>")
    except Exception:
        continue
print(f"Prepared {len(dd_lines)} DailyDialog conversations.")

print("\n=== Step 6: Merging, Shuffling & Exporting ===")
combined_corpus = keng_boosted + ts_lines + wiki_lines + scb_lines + dd_lines

# Shuffle dataset entries so training is evenly distributed across English, Thai, and Keng data
random.seed(42)
random.shuffle(combined_corpus)

output_file = '/content/train_combined.txt'
with open(output_file, 'w', encoding='utf-8') as f:
    for entry in combined_corpus:
        f.write(entry + '\n')

file_size_mb = os.path.getsize(output_file) / (1024 * 1024)
print(f"\nDataset construction complete!")
print(f"Total samples: {len(combined_corpus):,}")
print(f"Output File: {output_file} ({file_size_mb:.2f} MB)")

=== Step 1: Loading & Boosting Keng's Data ===
Loaded 212 Keng lines -> Boosted 10x to 2120 total entries.

=== Step 2: Processing TinyStories ===
Prepared 100000 TinyStories entries.

=== Step 3: Processing Thai Wiki ===
Prepared 50000 Thai Wiki entries.

=== Step 4: Processing SCB Thai-English ===
Prepared 100000 SCB entries (English & Thai).

=== Step 5: Processing DailyDialog ===
Prepared 1000 DailyDialog conversations.

=== Step 6: Merging, Shuffling & Exporting ===

Dataset construction complete!
Total samples: 253,120
Output File: /content/train_combined.txt (154.69 MB)


In [5]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

# 1. Initialize Byte-Level BPE Tokenizer
tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()

# 2. Configure Trainer (Vocab = 4096)
trainer = trainers.BpeTrainer(
    vocab_size=4096,
    special_tokens=["<|endoftext|>"]
)

# 3. Train on your text dataset
tokenizer.train(["/content/train_combined.txt"], trainer)

# 4. Save to JSON
tokenizer.save("/content/keng_tokenizer.json")
print("Tokenizer recreated and saved to /content/keng_tokenizer.json!")

Tokenizer recreated and saved to /content/keng_tokenizer.json!


Training

In [4]:
import numpy as np
import os
from tokenizers import Tokenizer
import gc

# 1. Load Tokenizer
tokenizer = Tokenizer.from_file("/content/keng_tokenizer.json")

# 2. Tokenize in Chunks to File (RAM Safe)
bin_path = "/content/train_data.bin"
raw_text_path = "/content/train_combined.txt"

print("Tokenizing combined dataset to memory-mapped binary file...")

token_buffer = []
chunk_size = 5000  # Process 5,000 lines per batch

with open(bin_path, "wb") as f_out:
    with open(raw_text_path, "r", encoding="utf-8") as f_in:
        lines = []
        for line in f_in:
            lines.append(line)
            if len(lines) >= chunk_size:
                # Encode batch of lines
                encoded = tokenizer.encode_batch(lines)
                for item in encoded:
                    token_buffer.extend(item.ids)

                # Write to disk as 16-bit integers (max vocab 4096 fits in uint16)
                np.array(token_buffer, dtype=np.uint16).tofile(f_out)
                token_buffer.clear()
                lines.clear()

        # Flush remaining lines
        if lines:
            encoded = tokenizer.encode_batch(lines)
            for item in encoded:
                token_buffer.extend(item.ids)
            np.array(token_buffer, dtype=np.uint16).tofile(f_out)
            token_buffer.clear()

gc.collect()
file_size_mb = os.path.getsize(bin_path) / (1024 * 1024)
print(f"Tokenization complete! Saved to {bin_path} ({file_size_mb:.2f} MB)")

Exception: No such file or directory (os error 2)

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class RWKV5_TimeMix(nn.Module):
    def __init__(self, n_embd, layer_id, n_layer):
        super().__init__()
        self.n_embd = n_embd
        self.layer_id = layer_id

        self.time_decay = nn.Parameter(torch.empty(n_embd))
        self.time_first = nn.Parameter(torch.empty(n_embd))
        self.time_mix_k = nn.Parameter(torch.empty(1, 1, n_embd))
        self.time_mix_v = nn.Parameter(torch.empty(1, 1, n_embd))
        self.time_mix_r = nn.Parameter(torch.empty(1, 1, n_embd))
        self.time_mix_g = nn.Parameter(torch.empty(1, 1, n_embd))

        self.key = nn.Linear(n_embd, n_embd, bias=False)
        self.value = nn.Linear(n_embd, n_embd, bias=False)
        self.receptance = nn.Linear(n_embd, n_embd, bias=False)
        self.gate = nn.Linear(n_embd, n_embd, bias=False)
        self.output = nn.Linear(n_embd, n_embd, bias=False)
        self.ln_x = nn.GroupNorm(1, n_embd)

        nn.init.uniform_(self.time_decay, -6, -1)
        nn.init.uniform_(self.time_first, -3, 0)
        nn.init.zeros_(self.time_mix_k)
        nn.init.zeros_(self.time_mix_v)
        nn.init.zeros_(self.time_mix_r)
        nn.init.zeros_(self.time_mix_g)

    def forward(self, x):
        B, T, C = x.size()
        xx = F.pad(x, (0, 0, 1, -1))

        xk = x * self.time_mix_k + xx * (1 - self.time_mix_k)
        xv = x * self.time_mix_v + xx * (1 - self.time_mix_v)
        xr = x * self.time_mix_r + xx * (1 - self.time_mix_r)
        xg = x * self.time_mix_g + xx * (1 - self.time_mix_g)

        k = self.key(xk)
        v = self.value(xv)
        r = torch.sigmoid(self.receptance(xr))
        g = F.silu(self.gate(xg))

        w = torch.exp(-torch.exp(self.time_decay)).view(1, 1, C)
        u = self.time_first.view(1, 1, C)

        wkv = torch.zeros(B, T, C, device=x.device)
        state = torch.zeros(B, C, device=x.device)
        for t in range(T):
            kt, vt = k[:, t, :], v[:, t, :]
            wkv[:, t, :] = state + u * kt * vt
            state = w * state + kt * vt

        x_out = self.ln_x(wkv.view(B * T, C)).view(B, T, C)
        return self.output(r * x_out * g)

class RWKV5_ChannelMix(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.time_mix_k = nn.Parameter(torch.zeros(1, 1, n_embd))
        self.time_mix_r = nn.Parameter(torch.zeros(1, 1, n_embd))
        self.key = nn.Linear(n_embd, n_embd * 3, bias=False)
        self.receptance = nn.Linear(n_embd, n_embd, bias=False)
        self.value = nn.Linear(n_embd * 3, n_embd, bias=False)

    def forward(self, x):
        xx = F.pad(x, (0, 0, 1, -1))
        xk = x * self.time_mix_k + xx * (1 - self.time_mix_k)
        xr = x * self.time_mix_r + xx * (1 - self.time_mix_r)

        k = torch.square(torch.relu(self.key(xk)))
        kv = self.value(k)
        r = torch.sigmoid(self.receptance(xr))
        return r * kv

class Block(nn.Module):
    def __init__(self, n_embd, layer_id, n_layer):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.att = RWKV5_TimeMix(n_embd, layer_id, n_layer)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffn = RWKV5_ChannelMix(n_embd)

    def forward(self, x):
        x = x + self.att(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

class KengGPT_RWKV(nn.Module):
    def __init__(self, vocab_size=4096, n_embd=384, n_layer=8):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.ModuleList([Block(n_embd, i, n_layer) for i in range(n_layer)])
        self.ln_out = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.head.weight = self.emb.weight  # Tied embeddings

    def forward(self, idx):
        x = self.emb(idx)
        for block in self.blocks:
            x = block(x)
        x = self.ln_out(x)
        return self.head(x)

# Instantiate 22M Parameter Model
model = KengGPT_RWKV().cuda()
params = sum(p.numel() for p in model.parameters())
print(f"KengGPT RWKV-v5 parameter count: {params / 1e6:.2f} Million")

KengGPT RWKV-v5 parameter count: 15.77 Million


In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from tqdm.auto import tqdm

# ---------------------------------------------------------
# 1. Dataset & DataLoader (Maxed out Batch Size)
# ---------------------------------------------------------
dataset = MemMapDataset("/content/train_data.bin", seq_len=128)

# Increased batch size to 256 to fully utilize your 15GB VRAM
loader = DataLoader(
    dataset,
    batch_size=256,   # <-- Increased from 64 to 256
    shuffle=True,
    drop_last=True,
    pin_memory=True,
    num_workers=0
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = KengGPT_RWKV().to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)

# Modern PyTorch 2.x Mixed Precision Scaler
scaler = torch.amp.GradScaler('cuda')

epochs = 2

print(f"Starting High-Speed FP16 Training with Batch Size 256 on {device}...")
model.train()

for epoch in range(epochs):
    total_loss = 0.0
    optimizer.zero_grad()

    progress_bar = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}")

    for batch_idx, (x, y) in enumerate(progress_bar):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

        with torch.amp.autocast('cuda', dtype=torch.float16):
            logits = model(x)
            loss = F.cross_entropy(logits.view(-1, 4096), y.view(-1))

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

        current_loss = loss.item()
        total_loss += current_loss
        progress_bar.set_postfix({"loss": f"{current_loss:.4f}"})

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch + 1}/{epochs} Complete | Avg Loss: {avg_loss:.4f}")

# Save Checkpoint
torch.save(model.state_dict(), "/content/kenggpt_rwkv_22m.pth")
print("Training complete! Model saved to /content/kenggpt_rwkv_22m.pth")

Starting High-Speed FP16 Training with Batch Size 256 on cuda...


Epoch 1/2:   0%|          | 0/1207 [00:00<?, ?it/s]

Epoch 1/2 Complete | Avg Loss: 6.6453


Epoch 2/2:   0%|          | 0/1207 [00:00<?, ?it/s]

Epoch 2/2 Complete | Avg Loss: 3.1847
Training complete! Model saved to /content/kenggpt_rwkv_22m.pth


quantization & testing



In [2]:
import torch
import torch.nn.functional as F
from tokenizers import Tokenizer, decoders

tokenizer = Tokenizer.from_file("/content/keng_tokenizer.json")
tokenizer.decoder = decoders.ByteLevel()
eos_id = tokenizer.token_to_id("<|endoftext|>")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = KengGPT_RWKV(vocab_size=4096, n_embd=384, n_layer=8).to(device)
model.load_state_dict(torch.load("/content/kenggpt_rwkv_22m.pth", map_location=device))
model.eval()

def generate_text(prompt, max_new_tokens=80, temperature=0.7, top_p=0.85, min_tokens=15):
    tokens = tokenizer.encode(prompt).ids
    input_ids = torch.tensor([tokens], dtype=torch.long, device=device)
    prompt_len = input_ids.shape[1]

    with torch.no_grad():
        for step in range(max_new_tokens):
            idx_cond = input_ids[:, -128:]
            logits = model(idx_cond)
            next_token_logits = logits[0, -1, :] / max(temperature, 1e-4)

            # 1. Suppress <|endoftext|> for the first `min_tokens` steps
            if eos_id is not None and step < min_tokens:
                next_token_logits[eos_id] = -float('Inf')

            # Clean invalid values
            next_token_logits = torch.nan_to_num(next_token_logits, nan=-1e9, posinf=1e4, neginf=-1e9)

            # Nucleus (Top-p) Sampling
            sorted_logits, sorted_indices = torch.sort(next_token_logits, descending=True)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

            sorted_indices_to_remove = cumulative_probs > top_p
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = 0

            indices_to_remove = sorted_indices[sorted_indices_to_remove]
            next_token_logits[indices_to_remove] = -float('Inf')

            probs = F.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

            # Stop if EOS is generated AFTER min_tokens
            if eos_id is not None and next_token.item() == eos_id:
                break

            input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=1)

    # Return only newly generated text
    generated_ids = input_ids[0][prompt_len:].tolist()
    return tokenizer.decode(generated_ids)

# Test Prompts
print("--- Test 1 (Burger Prompt with EOS Suppression) ---")
print("Keng Says : ",generate_text("", max_new_tokens=110, min_tokens=20))


Exception: No such file or directory (os error 2)

In [39]:
import torch
import torch.nn.functional as F
from tokenizers import Tokenizer, decoders

# 1. Load Tokenizer & Model Architecture
tokenizer = Tokenizer.from_file("/content/keng_tokenizer.json")
tokenizer.decoder = decoders.ByteLevel()
eos_id = tokenizer.token_to_id("<|endoftext|>")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize and load model
model = KengGPT_RWKV(vocab_size=4096, n_embd=384, n_layer=8).to(device)
model.load_state_dict(torch.load("/content/kenggpt_rwkv_22m.pth", map_location=device))
model.eval()

# 2. Flexible Interactive Generation Function
def chat_with_keng(
    user_input,
    system_prompt="You are Keng, a funny Discord chat bot.",
    max_new_tokens=60,
    min_tokens=10,
    temperature=0.4,
    top_p=0.8
):
    # Construct structured prompt template
    if system_prompt:
        full_prompt = f"System: {system_prompt}\nUser: {user_input}\nKeng:"
    else:
        full_prompt = f"User: {user_input}\nKeng:"

    tokens = tokenizer.encode(full_prompt).ids
    input_ids = torch.tensor([tokens], dtype=torch.long, device=device)
    prompt_len = input_ids.shape[1]

    with torch.no_grad():
        for step in range(max_new_tokens):
            # Maintain sliding context window matching training length
            idx_cond = input_ids[:, -128:]
            logits = model(idx_cond)

            # Apply temperature scaling
            next_token_logits = logits[0, -1, :] / max(temperature, 1e-4)

            # EOS suppression for early steps
            if eos_id is not None and step < min_tokens:
                next_token_logits[eos_id] = -float('Inf')

            # Clean numerical instabilities
            next_token_logits = torch.nan_to_num(next_token_logits, nan=-1e9, posinf=1e4, neginf=-1e9)

            # Top-p (nucleus) filtering
            sorted_logits, sorted_indices = torch.sort(next_token_logits, descending=True)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

            sorted_indices_to_remove = cumulative_probs > top_p
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = 0

            indices_to_remove = sorted_indices[sorted_indices_to_remove]
            next_token_logits[indices_to_remove] = -float('Inf')

            # Sample next token
            probs = F.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

            if eos_id is not None and step >= min_tokens and next_token.item() == eos_id:
                break

            input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=1)

    # Decode only the generated response portion
    generated_ids = input_ids[0][prompt_len:].tolist()
    return tokenizer.decode(generated_ids)

# ---------------------------------------------------------
# 3. Playground Area — Change your values here!
# ---------------------------------------------------------

SYSTEM_PROMPT = ""
USER_INPUT = "what is the derivative of 3x+x^2"

output = chat_with_keng(
    user_input=USER_INPUT,
    system_prompt=SYSTEM_PROMPT,
    max_new_tokens=110,    # Max response length
    min_tokens=10,         # Minimum tokens before allowing EOS
    temperature=0.35,      # Lower = more focused/deterministic, Higher = more chaotic
    top_p=0.75             # Nucleus sampling cut-off
)

print(f"Prompt: {USER_INPUT}")
print(f"Keng: {output}")

Prompt: what is the derivative of 3x+x^2
Keng:  Yes Warranty (Year): 1 Other: Yestern older in the first time, but it was too late. 


In [1]:
import torch
import torch.nn.functional as F
from tokenizers import Tokenizer, decoders

# 1. Load Tokenizer & Attach ByteLevel Decoder
tokenizer = Tokenizer.from_file("/content/keng_tokenizer.json")
tokenizer.decoder = decoders.ByteLevel()
eos_id = tokenizer.token_to_id("<|endoftext|>")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize and load model weights
model = KengGPT_RWKV(vocab_size=4096, n_embd=384, n_layer=8).to(device)
model.load_state_dict(torch.load("/content/kenggpt_rwkv_22m.pth", map_location=device))
model.eval()

# 2. Optimized Inference Function with Repetition Penalty
def generate_text(
    prompt,
    max_new_tokens=80,
    min_tokens=15,
    temperature=0.35,
    top_p=0.75,
    repetition_penalty=1.2
):
    tokens = tokenizer.encode(prompt).ids
    input_ids = torch.tensor([tokens], dtype=torch.long, device=device)
    prompt_len = input_ids.shape[1]

    with torch.no_grad():
        for step in range(max_new_tokens):
            # Crop context window to the last 128 tokens (matches training sequence length)
            idx_cond = input_ids[:, -128:]
            logits = model(idx_cond)

            # Extract raw logits for the last token position
            next_token_logits = logits[0, -1, :]

            # --- REPETITION PENALTY IMPLEMENTATION ---
            # Penalize tokens that have already been generated to break infinite loops
            for token_id in set(input_ids[0].tolist()):
                if next_token_logits[token_id] > 0:
                    next_token_logits[token_id] /= repetition_penalty
                else:
                    next_token_logits[token_id] *= repetition_penalty
            # ----------------------------------------

            # Apply temperature scaling
            next_token_logits = next_token_logits / max(temperature, 1e-4)

            # Suppress <|endoftext|> for the first `min_tokens` steps
            if eos_id is not None and step < min_tokens:
                next_token_logits[eos_id] = -float('Inf')

            # Clean numerical instabilities (NaN / Inf protection)
            next_token_logits = torch.nan_to_num(next_token_logits, nan=-1e9, posinf=1e4, neginf=-1e9)

            # Top-p (nucleus) filtering
            sorted_logits, sorted_indices = torch.sort(next_token_logits, descending=True)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

            sorted_indices_to_remove = cumulative_probs > top_p
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = 0

            indices_to_remove = sorted_indices[sorted_indices_to_remove]
            next_token_logits[indices_to_remove] = -float('Inf')

            # Sample next token
            probs = F.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

            # Exit loop if EOS token is generated AFTER min_tokens threshold
            if eos_id is not None and step >= min_tokens and next_token.item() == eos_id:
                break

            input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=1)

    # Decode and return only the newly generated text
    generated_ids = input_ids[0][prompt_len:].tolist()
    return tokenizer.decode(generated_ids)

# ---------------------------------------------------------
# 3. Interactive Testing Playground
# ---------------------------------------------------------

prompts = [
    "ผมชื่อเก่ง"]

for p in prompts:
    print(f"\n--- Prompt: {p} ---")
    output = generate_text(
        prompt=p,
        max_new_tokens=60,
        min_tokens=10,
        temperature=0.35,
        top_p=0.75,
        repetition_penalty=1.2
    )
    print(f"Output: {output}")

Exception: No such file or directory (os error 2)